# MapReduce Mini‑Projects 



This notebook contains **five short tasks** that mimic MapReduce using plain Python.
Each task has a starter cell. Write your code where indicated.

You can run the final **self‑tests** at the end of each task to check your work.

In [ ]:

# ---- Sample Datasets (reuse across tasks) ----

# Tiny web pages corpus
web_pages = [
    "MapReduce makes big data processing scalable",
    "Map reduce reduces the complexity of processing",
    "Big data needs big ideas",
]

# LinkedIn-like view logs: (user_id, content_type, content_id, timestamp)
logs = [
    ("u1", "profile", "p10", "2025-08-20 09:00:01"),
    ("u2", "post",    "c42", "2025-08-20 09:00:05"),
    ("u1", "post",    "c42", "2025-08-20 09:01:10"),
    ("u3", "article", "a7",  "2025-08-20 09:02:00"),
    ("u2", "profile", "p10", "2025-08-20 09:03:10"),
    ("u4", "video",   "v3",  "2025-08-20 09:03:21"),
    ("u3", "post",    "c42", "2025-08-20 09:04:00"),
    ("u1", "video",   "v3",  "2025-08-20 09:05:10"),
    ("u5", "post",    "c9",  "2025-08-20 09:06:30"),
    ("u2", "video",   "v3",  "2025-08-20 09:07:11"),
]

# Profiles: user -> headline/skills text
profiles = {
    "u1": "Data Scientist | Python | Spark | ML",
    "u2": "Backend Engineer | Go | Kubernetes | Cloud",
    "u3": "ML Engineer | Python | MLOps | Cloud",
    "u4": "Frontend Developer | React | TypeScript",
    "u5": "Data Analyst | SQL | Python | BI",
}

# Social graph edges (connections are undirected)
edges = [
    ("u1","u2"), ("u2","u3"), ("u3","u4"), ("u1","u5"),
    # note: u2-u5 not connected, u4-u5 not connected
]


## Helper: Mini shuffle‑and‑sort
We'll use a helper that groups by key just like the shuffle phase.

In [ ]:

from collections import defaultdict

def shuffle_group(pairs):
    grouped = defaultdict(list)
    for k, v in pairs:
        grouped[k].append(v)
    return grouped


## Task 1 — Word Count with Stop‑Words
Count word frequency across `web_pages` (ignore case). Remove common stop‑words: `{'the','and','of','a','an','to','is','big'}`.
**Implement**: `map_words`, `reduce_counts`.

In [ ]:

stop = {'the','and','of','a','an','to','is','big'}

def map_words(pages):
    # return list of (word, 1)
    pairs = []
    # YOUR CODE HERE
    return pairs

def reduce_counts(grouped):
    # grouped: dict[word] -> list of 1s
    final = {}
    # YOUR CODE HERE
    return final

# --- tests ---
pairs = map_words(web_pages)
grouped = shuffle_group(pairs)
wc = reduce_counts(grouped)
print(sorted(wc.items())[:10])


## Task 2 — Views per Content Type
Compute number of views for each `content_type` (profile/post/article/video) from `logs`.
**Implement**: `map_types`, `reduce_sum`.

In [ ]:

def map_types(rows):
    # emit (content_type, 1)
    pairs = []
    # YOUR CODE HERE
    return pairs

def reduce_sum(grouped):
    # sum list of numbers per key
    out = {}
    # YOUR CODE HERE
    return out

# tests
pairs = map_types(logs)
grouped = shuffle_group(pairs)
views_by_type = reduce_sum(grouped)
views_by_type


## Task 3 — Top‑K Content by Views (K=2)
Count views per `content_id` and print the top 2 most‑viewed content IDs.
**Implement**: `map_content`, `top_k`.

In [ ]:

def map_content(rows):
    pairs = []
    # YOUR CODE HERE: emit (content_id, 1)
    return pairs

def top_k(counts, k=2):
    # counts: dict[content_id] -> count
    # return list of (content_id, count) sorted desc by count
    # YOUR CODE HERE
    return []

# tests
pairs = map_content(logs)
grouped = shuffle_group(pairs)
counts = {k: sum(vs) for k, vs in grouped.items()}
top_k(counts, 2)


## Task 4 — Inverted Index of Skills → Users
From `profiles`, build an inverted index mapping each **skill/word** to list of **user IDs** containing it.
Normalize to lowercase and keep only alpha tokens (letters).
**Implement**: `map_profile_tokens`, `reduce_unique_users`.

In [ ]:

import re

def tokenize(text):
    return [t.lower() for t in re.findall(r"[A-Za-z]+", text)]

def map_profile_tokens(profiles):
    pairs = []
    # YOUR CODE HERE: for each user and token -> emit (token, user)
    return pairs

def reduce_unique_users(grouped):
    out = {}
    # YOUR CODE HERE: make unique user list per token, sorted
    return out

# tests
pairs = map_profile_tokens(profiles)
grouped = shuffle_group(pairs)
index = reduce_unique_users(grouped)
{k: index[k] for k in list(index)[:5]}


## Task 5 — Friend‑of‑Friend Recommendations (Mutuals)
Using undirected `edges`, recommend new connections based on number of **mutual friends**.
For each pair of unconnected users `(a,b)` that share a neighbor `x`, count 1 mutual. Output top suggestions per user.
**Implement**: `map_mutuals`, `reduce_counts`, `suggestions_for`.

In [ ]:

from collections import defaultdict

def neighbors(edges):
    nb = defaultdict(set)
    for a,b in edges:
        nb[a].add(b); nb[b].add(a)
    return nb

def map_mutuals(edges):
    # For each user x, for each unordered pair (a,b) in neighbors(x),
    # emit ((a,b), 1) to count one mutual friend x.
    pairs = []
    # YOUR CODE HERE
    return pairs

def reduce_counts(grouped):
    return {k: sum(vs) for k,vs in grouped.items()}

def suggestions_for(edges, topn=2):
    nb = neighbors(edges)
    # count mutuals
    pairs = map_mutuals(edges)
    counts = reduce_counts(shuffle_group(pairs))
    # Build per-user suggestions skipping already-connected pairs
    sugg = defaultdict(list)
    for (a,b), c in counts.items():
        if b not in nb[a]:  # ensure not already friends
            sugg[a].append((b,c))
        if a not in nb[b]:
            sugg[b].append((a,c))
    # sort each list by count desc then id
    for u in list(sugg):
        sugg[u] = sorted(sugg[u], key=lambda x: (-x[1], x[0]))[:topn]
    return dict(sugg)

# tests
suggestions_for(edges, topn=2)
